# Magnitude Processing

This notebook shows how the harmonic transposition is done in order to match the target note.

In [1]:
# Enabling ipympl for interactive plots and styling the UI a bit
# get_ipython().run_line_magic('matplotlib', 'widget') # ipympl
get_ipython().run_cell_magic('html', '', '<style>.widget-readout { color: white; font-size: 1.2em; box-shadow: none !important ;} </style>') # style

import sys, os, datetime
import numpy as np
import pyaudio as pa
import matplotlib.pyplot as plt

sys.path.append('/sms-tools/software/models')
import utilFunctions as UF
import sineModel as SM
# import stochasticModel as STM

sys.path.append('../')
from utils.structures import Sound
from scipy.signal import resample, blackmanharris, triang, hanning
from scipy.fftpack import fft, ifft, fftshift
from scipy.io.wavfile import write
from ipywidgets import ( HTML, Layout, FloatSlider, Label, ToggleButton, Button, GridBox, interactive_output, interact )

# Constants
DEFAULT_DATA_FOLDER = '../../data'
DEFAULT_SOUNDS_FOLDER = DEFAULT_DATA_FOLDER + '/sounds'
DEFAULT_SOUND_FILE_1 = DEFAULT_SOUNDS_FOLDER + '/violin-B3.wav'
DEFAULT_SOUND_FILE_2 = DEFAULT_SOUNDS_FOLDER + '/soprano-E4.wav'
ANALYSIS_OUTPUT_FOLDER = DEFAULT_DATA_FOLDER + '/analysis_output'
MORPHINGS_OUTPUT_FOLDER = DEFAULT_DATA_FOLDER + 'morphing_output_test'

NS = 512 # size of fft used in synthesis
H = int(NS/4) # hop size (has to be 1/4 of NS) - 128
NUMBER_OF_FFT_SYNTH_FRAMES = 4 # number of FFT synth frames
FRAME_TO_PLOT = 0 # 100 # frame to plot
NUMBER_OF_FRAMES_TO_PLOT  = 8 # number of frames to plots
GENERATE_PLOTS = False # generate the plots

DARK_MODE = False

if (DARK_MODE):
    params = {
        "text.color" : "w",
        "ytick.color" : "w",
        "xtick.color" : "w",
        "axes.labelcolor" : "w",
        "axes.edgecolor" : "w",
#         "axes.facecolor" : 'e5e5e5'
    }
    plt.rcParams.update(params)

PATH:  /Users/Marc/Research/Tools/sms-tools/software/models/./utilFunctions_C/


In [3]:
# Load the sounds by default
sound_1 = Sound(DEFAULT_SOUND_FILE_1)
sound_2 = Sound(DEFAULT_SOUND_FILE_2)

# Loading the .had files
sound_1.load_had_file()
sound_2.load_had_file()

# Empty sound where the morph will be generated 
sound_morph = Sound(DEFAULT_SOUND_FILE_2)

In [ ]:
%matplotlib inline
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np

def magnitude_processing(cycles_per_second, b):
    
    plt.figure(figsize=(20,15))
    mags_1 = sound_1.analysis.output.values.hmag
    
    highest_mag_1 = max(mags_1.min(), mags_1.max(), key=abs);
    
    # Recalculate the harmonics for the current midi note
    for i in range(0, len(mags), 1):
        
        # Fundamental note of this frame (in Hz)
        current_fundamental = mags[i][0];
    
        for j in range(0, len(mags[i]), 1):
            mags[i][j] = (mags[i][j] / current_fundamental) * cycles_per_second
                
    mags[ mags==0 ] = np.nan

    plt.plot(mags)
    plt.ylim(0, 15000)
    plt.show()

interactive_plot = interact(magnitude_processing, cycles_per_second=(10.0, 1000.0), b=(-3, 3, 0.5))
# interactive_plot = interact_manual(f, cycles_per_second=(10.0, 1000.0), b=(-3, 3, 0.5))
# output = interactive_plot.children[-1]
# output.layout.height = '600px'
interactive_plot

In [77]:
test = np.array([-41.81383135625492, -34.43317433640501, -38.765882102408646, -53.953922996792606, -57.95119838902268, -61.76039823869481, -62.440455348293796, -100.0, -100.0, -59.54360619121134, -65.74963906370155, -73.89473083553031, -54.07063226474117, -56.6164591853808, -60.75736238720696, -79.06166300458122, -75.54659345357427, -74.24948679749816, -79.50656785954081, -83.90416609118641, -72.876456571967, -79.3608446633624, -77.76089131251554, -71.09183578009291, -70.70602387192835, -79.61236278423888, -78.34245988201059, -82.63040954318654, -89.64560316746693, -100.0, -99.17190697056431, -99.17190697056431, -97.19452278096777, -89.2478985501205, -92.95971261213498, -98.48676581386135, -96.3002058568953, -100.0, -94.54533826097419, -98.57668399214931, -96.8790207958873, -95.74500193786761, -93.38518285376459, -96.02121951667587, -99.21878075071817, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0, -100.0])

In [78]:
min(test), max(test)

(-100.0, -34.43317433640501)

In [79]:
def normalize(x):
    min_val = min(x)
    max_val = max(x)
    a = -100  
    b = 0
#     x = (b - a) * ( (x-(min_val*x)) / ((max_val*x)-(min_val*x)) ) + a
    x = (b - a) * ( (x-min_val) / (max_val-min_val) ) + a
    return x

In [80]:
normalize(test)

array([ -11.25669414,    0.        ,   -6.6080792 ,  -29.77229485,
        -35.86878549,  -41.67843056,  -42.7156275 , -100.        ,
       -100.        ,  -38.29746461,  -47.76266719,  -60.1852478 ,
        -29.95029534,  -33.83309261,  -40.14863886,  -68.06565396,
        -62.70460511,  -60.72630794,  -68.74420573,  -75.45125336,
        -58.63221507,  -68.52195431,  -66.08176702,  -55.91038009,
        -55.32195461,  -68.90556008,  -66.96875303,  -73.50856888,
        -84.20787231, -100.        ,  -98.73702437,  -98.73702437,
        -95.72119408,  -83.60130853,  -89.26242453,  -97.69207344,
        -94.35721631, -100.        ,  -91.68075977,  -97.82921318,
        -95.24000259,  -93.51044065,  -89.91133538,  -93.93171708,
        -98.80851446, -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.     

In [74]:
test

array([ -41.81383136,  -34.43317434,  -38.7658821 ,  -53.953923  ,
        -57.95119839,  -61.76039824,  -62.44045535, -100.        ,
       -100.        ,  -59.54360619,  -65.74963906,  -73.89473084,
        -54.07063226,  -56.61645919,  -60.75736239,  -79.061663  ,
        -75.54659345,  -74.2494868 ,  -79.50656786,  -83.90416609,
        -72.87645657,  -79.36084466,  -77.76089131,  -71.09183578,
        -70.70602387,  -79.61236278,  -78.34245988,  -82.63040954,
        -89.64560317, -100.        ,  -99.17190697,  -99.17190697,
        -97.19452278,  -89.24789855,  -92.95971261,  -98.48676581,
        -96.30020586, -100.        ,  -94.54533826,  -98.57668399,
        -96.8790208 ,  -95.74500194,  -93.38518285,  -96.02121952,
        -99.21878075, -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.        ,
       -100.        , -100.        , -100.        , -100.     